# NL experiment · 03 The spatial folds of the IWI study, with the NL target

This study keeps the five spatial folds A–E of `03_Spatial_Folds` of the IWI study unchanged: the same clusters, the same spatial groups (clusters closer than 9.50 km), the same folds and the same clusters left out between groups. So every run of 04–15 here has the same training, validation and test clusters as the run of the same model, input and test fold in the IWI study, and the two can be compared cluster by cluster.

This notebook joins the target of `02_NL_Target` to `train_clusters_folds.csv` of the IWI study and writes **`train_clusters_folds_nl.csv`** to `WORK_DIR`: its rows in the same order, with the columns of 02 (`nl`, `nl_raw`, `nl_norm_mean`, `nl_norm_sd`, …). 04–18 read this file (`FOLDS_CSV`). The band caches of the IWI study hold the clusters in the order of its folds file (per period, sorted by `cluster_id`), so the model notebooks of this study use them as they are, and they check that the two files still list the same clusters.

**How the target spreads over the folds.** 03 of the IWI study balanced the clusters of every country over the folds (15–25 % per fold) and spread each country's poorest, middle and richest third of IWI over at least three folds. NL and IWI are related but not the same, so the same check is made here for the NL target, without changing the folds:

* **NL parts**: the clusters of every country and NL period (DMSP up to 2013, VIIRS from 2014: the two periods are on different scales) are cut into `N_STRATA` equal parts by the target (the darkest, middle and brightest third; fewer parts below `MIN_STRATUM` clusters per part). For every part: clusters per fold, the largest fold share (above `STRATUM_MAX` = 35 % the part counts as crowded, the limit of the IWI parts in 03), and the smallest share of the part that lies in the training folds of a test fold that holds some of it (0 would mean a test fold whose part no model has seen).
* **The target per fold**: mean, standard deviation and percentiles of the target and of `nl_raw`, per period; the countries whose folds differ most in their mean target; the distribution of the target in every fold.

A crowded part is reported, not repaired: the folds are those of the IWI study. If the folds of the IWI study change (03 run again), run 02 and this notebook again, and train 04–15 anew in new run folders.

**Output** in `WORK_DIR`: `train_clusters_folds_nl.csv`, `folds_by_nl_stratum.csv`, `folds_nl_summary.csv`, `folds_nl_target.png`.

# Import libraries

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Settings

In [ ]:
# ---- input / output ------------------------------------------------------------------------
DATA_DIR      = '/path/to/data/'   # the IWI study (01-03, caches)
WORK_DIR      = os.path.join(DATA_DIR, 'WaveletNL')                   # this study
IWI_FOLDS_CSV = os.path.join(DATA_DIR, 'train_clusters_folds.csv')    # written by 03_Spatial_Folds of the IWI study
TARGET_CSV    = os.path.join(WORK_DIR, 'train_clusters_nl.csv')       # written by 02_NL_Target
OUT_CSV       = os.path.join(WORK_DIR, 'train_clusters_folds_nl.csv') # IWI_FOLDS_CSV + the target columns of 02
STRATUM_CSV   = os.path.join(WORK_DIR, 'folds_by_nl_stratum.csv')
SUMMARY_CSV   = os.path.join(WORK_DIR, 'folds_nl_summary.csv')
PLOT_PNG      = os.path.join(WORK_DIR, 'folds_nl_target.png')
BAND_CACHES   = [os.path.join(DATA_DIR, d) for d in ('table1_cache', 'table2_cache', 'dft_cache')]   # of 04-15

TARGET = 'nl'
FOLDS  = ['A', 'B', 'C', 'D', 'E']

# ---- the check of section 3, with the limits of the IWI parts of 03_Spatial_Folds
N_STRATA    = 3        # per country and NL period, the clusters are cut into this many equal parts by the target
MIN_STRATUM = 30       # every part needs at least this many clusters; smaller groups get fewer parts
STRATUM_MAX = 0.35     # a part with more than this share in one fold is crowded

TARGET_COLS = ['nl', 'nl_raw', 'nl_norm_mean', 'nl_norm_sd', 'nl_raw_unclipped', 'nl_max', 'nl_dark_share',
               'nl_clipped_share', 'nl_period', 'nl_file_sensor', 'nl_error']

# 1. The folds of the IWI study and the target of 02

The rows of `train_clusters_folds.csv` stay as they are, in their order; the target columns of 02 are joined on `cluster_id`. A cluster of the folds without a row in 02 (02 was run on an older `train_clusters.csv`) stops the notebook: run 02 again.

In [ ]:
folds = pd.read_csv(IWI_FOLDS_CSV, dtype={'cluster_id': str})
folds = folds.rename(columns={'ClusterID': 'cluster_id', 'year.of.interview': 'year',
                              'month.of.interview': 'month', 'iwi_avg': 'iwi'})
target = pd.read_csv(TARGET_CSV, dtype={'cluster_id': str})
assert folds['cluster_id'].is_unique and target['cluster_id'].is_unique, 'a cluster_id appears twice'
missing = sorted(set(folds['cluster_id']) - set(target['cluster_id']))
assert not missing, (f'{len(missing)} clusters of {IWI_FOLDS_CSV} are not in {TARGET_CSV} (e.g. {missing[:3]}): '
                     'run 02_NL_Target again')
keep = [c for c in TARGET_COLS if c in target]
data = folds.drop(columns=[c for c in keep if c in folds]).merge(target[['cluster_id'] + keep], on='cluster_id',
                                                                  how='left', validate='one_to_one')
assert data['cluster_id'].tolist() == folds['cluster_id'].tolist()      # the rows and their order of the IWI folds
if 'nl_sensor' not in data:
    data['nl_sensor'] = np.where(data['year'] >= 2014, 'VIIRS', 'DMSP')
data['fold'] = data['fold'].fillna('').astype(str)
has = data[TARGET].notna() & data['fold'].isin(FOLDS)
print(f"{len(data)} clusters in {data['country'].nunique()} countries, {has.sum()} with a target and a fold;",
      'without a target:', int(data[TARGET].isna().sum()), '| clusters of 02 not in the folds (left out by 03):',
      len(set(target['cluster_id']) - set(folds['cluster_id'])))
print(pd.crosstab(data['nl_period'], data['fold'], margins=True).to_string())

# 2. Save

`train_clusters_folds_nl.csv` is written under a temporary name and then renamed. When the file exists already and its folds differ (the IWI folds changed), the earlier one is kept as `train_clusters_folds_nl_until_<date>.csv`: every run made with it has to be trained again in a new run folder, because 04–15 skip a run whose `result.json` exists.

In [ ]:
os.makedirs(WORK_DIR, exist_ok=True)
if os.path.exists(OUT_CSV):
    old = pd.read_csv(OUT_CSV, usecols=['cluster_id', 'fold'], dtype=str).fillna('')
    if old['cluster_id'].tolist() != data['cluster_id'].tolist() or old['fold'].tolist() != data['fold'].tolist():
        stamp = time.strftime('%Y%m%d_%H%M', time.localtime(os.path.getmtime(OUT_CSV)))
        backup = OUT_CSV.replace('.csv', f'_until_{stamp}.csv')
        os.replace(OUT_CSV, backup)
        print(f'The folds changed. The earlier file is now {backup}: train 04-15 again in new run folders.')
tmp = OUT_CSV + '.tmp'
data.to_csv(tmp, index=False, float_format='%.10g')
os.replace(tmp, OUT_CSV)
print(f'{len(data)} clusters ->', OUT_CSV)

# the band caches of the IWI study (04-15 read them): their cluster lists must be those of the folds, per period
rows = []
for root in BAND_CACHES:
    for d in sorted(os.listdir(root)) if os.path.isdir(root) else []:
        index = os.path.join(root, d, 'index.csv')
        if not os.path.exists(index):
            continue
        period = d.split('_')[0]
        expect = sorted(data.loc[data['nl_sensor'] == period, 'cluster_id'])
        cached = pd.read_csv(index, usecols=['cluster_id'], dtype={'cluster_id': str})['cluster_id'].tolist()
        rows.append({'cache': os.path.relpath(os.path.join(root, d), DATA_DIR), 'clusters': len(cached),
                     'same clusters as the folds': cached == expect})
caches = pd.DataFrame(rows, columns=['cache', 'clusters', 'same clusters as the folds'])
if len(caches):
    display(caches)
    if not caches['same clusters as the folds'].all():
        print('WARNING: a cache lists other clusters than the folds: the notebook that uses it builds it again '
              '(also in the IWI study). Were the folds of the IWI study changed after the caches were built?')
else:
    print('no band cache found yet in', BAND_CACHES, '(the first model notebook builds them)')

# 3. How the NL target spreads over the folds

1. **NL parts**: per country and NL period, `N_STRATA` equal parts by the target (fewer for small groups; a group of fewer than 2 × `MIN_STRATUM` clusters is not cut). For every part: clusters per fold, the number of folds that hold it, the smallest and largest fold share, `ok` (largest share ≤ `STRATUM_MAX`), and `min_train_share_%`: the smallest share of the part in the three training folds of a test fold that holds some of it (validation fold = the fold before the test fold, as in 04–15).
2. **The target per fold**, per period: clusters, mean, standard deviation and percentiles of the target and the median of `nl_raw`; the urban share.
3. The countries whose folds differ most in mean target (per period), and the distribution of the target in every fold for all clusters and for the two country-periods with the largest difference.

In [ ]:
d = data[has].copy()
part = np.full(len(d), -1)
names, rows = [], []
for (country, period), g in d.groupby(['country', 'nl_period'], sort=True):
    k = int(min(N_STRATA, len(g) // MIN_STRATUM))
    if k < 2:
        continue
    cut = pd.qcut(g[TARGET].rank(method='first'), k, labels=False).to_numpy()   # equal parts (ties by order)
    for p in range(k):
        v = g[TARGET].to_numpy()[cut == p]
        names.append(f'{country} {period} nl {v.min():.2f} to {v.max():.2f}')
        part[d.index.get_indexer(g.index[cut == p])] = len(names) - 1
d['nl_part'] = part
print(f'{len(names)} NL parts in {d.loc[part >= 0, ["country", "nl_period"]].drop_duplicates().shape[0]} country-periods;',
      int((part < 0).sum()), 'clusters in no part (small country-periods)')

t = d[d['nl_part'] >= 0]
table = pd.crosstab(t['nl_part'], t['fold']).reindex(columns=FOLDS, fill_value=0)
table.index = [names[i] for i in table.index]
share = table.div(table.sum(axis=1), axis=0)
by_stratum = table.copy()
by_stratum.insert(0, 'clusters', table.sum(axis=1))
by_stratum['folds'] = (table > 0).sum(axis=1)
by_stratum['min_share_%'] = (100 * share.min(axis=1)).round(1)
by_stratum['max_share_%'] = (100 * share.max(axis=1)).round(1)
by_stratum['ok'] = share.max(axis=1) <= STRATUM_MAX + 1e-9
val = lambda f: (f - 1) % len(FOLDS)
by_stratum['min_train_share_%'] = [
    round(100 * min(row[[x for x in range(len(FOLDS)) if x not in (f, val(f))]].sum() / row.sum()
                    for f in range(len(FOLDS)) if row[f] > 0), 1) for row in table.to_numpy()]
by_stratum.index.name = 'part'
by_stratum.to_csv(STRATUM_CSV)
if len(by_stratum):
    print(f"NL parts with at most {100 * STRATUM_MAX:.0f} % in one fold: {by_stratum['ok'].sum()} of {len(by_stratum)} | "
          f"in all {len(FOLDS)} folds: {(by_stratum['folds'] == len(FOLDS)).sum()} | in at least 3 folds: "
          f"{(by_stratum['folds'] >= 3).sum()} | in the training folds of every test fold: at least "
          f"{by_stratum['min_train_share_%'].min()} % of every part")
    for s, r in by_stratum[~by_stratum['ok']].iterrows():
        print(f'   crowded: {s} ({r.clusters} clusters, largest fold share {r["max_share_%"]} %)')
print('->', STRATUM_CSV)
by_stratum.sort_values('max_share_%', ascending=False).head(15)

In [ ]:
# 2. the target per fold and period
urban = d['rural'].astype(str).str.strip().str.lower().isin(['false', '0', '0.0', 'u', 'urban']) if 'rural' in d \
    else pd.Series(False, index=d.index)


def fold_summary(g):
    q = g[TARGET].quantile([0.1, 0.25, 0.5, 0.75, 0.9]).to_numpy()
    return pd.Series({'clusters': len(g), 'countries': g['country'].nunique(), 'urban share': urban[g.index].mean(),
                      'nl mean': g[TARGET].mean(), 'nl sd': g[TARGET].std(), 'nl p10': q[0], 'nl p25': q[1],
                      'nl median': q[2], 'nl p75': q[3], 'nl p90': q[4], 'nl_raw median': g['nl_raw'].median(),
                      'iwi mean': g['iwi'].mean() if 'iwi' in g else np.nan})


cols = [TARGET, 'nl_raw', 'country'] + (['iwi'] if 'iwi' in d else [])
summary = pd.concat([d.assign(nl_period='all')[cols + ['nl_period', 'fold']], d[cols + ['nl_period', 'fold']]])
summary = summary.groupby(['nl_period', 'fold'])[cols].apply(fold_summary)
summary.to_csv(SUMMARY_CSV)
print('->', SUMMARY_CSV)
summary.round(3)

In [ ]:
# 3. the countries whose folds differ most, and the distribution of the target per fold
fold_mean = d.pivot_table(index=['country', 'nl_period'], columns='fold', values=TARGET, aggfunc='mean')
fold_mean = fold_mean.reindex(columns=FOLDS)
spread = (fold_mean.max(axis=1) - fold_mean.min(axis=1)).sort_values(ascending=False)
print('country-periods whose folds differ most in mean target (largest minus smallest fold mean):')
display(pd.concat([fold_mean.loc[spread.index[:10]].round(2), spread.head(10).round(2).rename('range')], axis=1))

FOLD_LINE = ['#2a78d6', '#1baf7a', '#eda100', '#008300', '#e34948']   # A-E as in the maps of 03
panels = [(f'all clusters, {p}', d[d['nl_period'] == p]) for p in sorted(d['nl_period'].unique())]
panels += [(f'{c} {p}', d[(d['country'] == c) & (d['nl_period'] == p)]) for c, p in spread.index[:2]]
fig, axes = plt.subplots(1, len(panels), figsize=(4.4 * len(panels), 3.8), constrained_layout=True, squeeze=False)
for ax, (title, g) in zip(axes[0], panels):
    for f, name in enumerate(FOLDS):
        v = np.sort(g.loc[g['fold'] == name, TARGET].to_numpy())
        if len(v):
            ax.step(v, np.arange(1, len(v) + 1) / len(v), where='post', color=FOLD_LINE[f], linewidth=1.5,
                    label=f'{name}  n={len(v)}')
    ax.set_title(title, loc='left', fontsize=10)
    ax.set_xlabel(f'target {TARGET}')
    ax.grid(color='#e1e0d9', linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    ax.legend(frameon=False, fontsize=7, loc='lower right')
axes[0, 0].set_ylabel("share of the fold's clusters with a target below")
fig.savefig(PLOT_PNG, dpi=150)
plt.show()
print('->', PLOT_PNG)